# 05 · AVA-VLA A100 1-step FFT · checkpoint 별도 프로세스 재로딩
이 노트북은 Canonical D/seed 42의 **Train Episode 1개만** 원본 TFDS에서 준비하고
AVA-VLA full-parameter forward/backward/update 1-step, 전체 checkpoint 저장과 별도 Python
프로세스 재로딩을 검사합니다. 14,347 Episode 전체 전처리·복사나 본 학습은 실행하지 않습니다.

기본 Colab Python 3.13은 사용하지 않고 `uv`로 Python 3.11 환경을 만듭니다.
`bitsandbytes`를 포함한 AVA 고정 의존성을 설치하므로 네트워크 설치 시간은 필요하지만
모델·데이터를 새로 다운로드하지 않습니다. 입력은 Drive의 기존 TFDS와 AVA 가중치입니다.

**AVA temporal 한계:** FFT 샘플에는 이전 query의 action feature가 없어
`temporal_context=None`인 learned zero-context 경로만 학습합니다. 따라서 원본 AVA의
순차 temporal-context 학습/TBPTT 또는 rollout 성능 검증과 동등하지 않습니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/real_drive')
ROOT = Path('/content/real_drive/MyDrive/VLADE')  # 네 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = "5b0a2a85b3b4b849c77855886ca5fa093b7048fd158b956ca9dc39e5614c55e7"
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'))
def run(args, env=None):
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('실행에 실패했습니다. 위 오류를 확인하십시오. 후속 학습은 시작하지 않습니다.')
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

In [ ]:
import tomllib
upstream = CODE / 'third_party/ava_vla'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(
    ('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    'transformers @ git+https://github.com/LiAuto-DSR/transformers-avavla.git@a03eee5da1794c45f5a911b60a9a4545e6e0ed1f',
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
AVA_PY = environment('vlade_ava_fft_py311', '3.11', dependencies)
run([UV, 'pip', 'install', '--python', AVA_PY, '--no-deps', '-e', upstream])
run([AVA_PY, '-c', "import bitsandbytes,torch,transformers,tensorflow_datasets; "
    "assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(); "
    "p=torch.cuda.get_device_properties(0); "
    "print(torch.__version__,transformers.__version__,bitsandbytes.__version__,p.name,p.total_memory/2**30)"])
run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(AVA_PY)], text=True)
(ROOT / 'ava_environment_freeze.txt').write_text(freeze)

In [ ]:
TFDS_ROOT = ROOT / 'data/processed/libero_plus_tfds'
PREPARED = Path('/content/vlade-preprocess/libero_plus_canonical_train1')
if not (PREPARED / 'COMPLETE').is_file():
    run([AVA_PY, 'scripts/data/prepare_single_canonical_episode.py',
         '--tfds-root', TFDS_ROOT, '--output', PREPARED,
         '--canonical-manifest', CODE / 'experiments/manifests/libero_plus_rlds/canonical_d_seed42.jsonl.gz',
         '--canonical-sha256', '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80',
         '--canonical-normalization', CODE / 'experiments/manifests/libero_plus_rlds/normalization_train_d_seed42.json',
         '--canonical-normalization-sha256', 'eac95bcc81b6182b90a2bd030096df6e3e408cb88923d0a1d57dc6c81e297c67'])
manifest = json.loads((PREPARED / 'manifest.json').read_text())
assert manifest['dataset_scope'] == 'canonical_train_episode_smoke'
assert manifest['counts'] == {'train': 1}
assert manifest['canonical_manifest_sha256'] == '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80'
assert manifest['canonical_normalization_sha256'] == 'eac95bcc81b6182b90a2bd030096df6e3e408cb88923d0a1d57dc6c81e297c67'
print(json.dumps({key: manifest[key] for key in (
    'dataset_scope', 'counts', 'selected_source_episode_index',
    'canonical_manifest_sha256', 'canonical_normalization_sha256')}, indent=2))
print('Episode:', manifest['episodes'][0]['episode_id'],
      'queries:', manifest['episodes'][0]['queries'])

In [ ]:
OUTPUT = ROOT / 'runs/ava_fft/ava-a100-one-step-seed42'
CONFIGS = ROOT / 'run-configs'; CONFIGS.mkdir(exist_ok=True)
CONFIG = CONFIGS / 'ava-a100-one-step-seed42.json'
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
config['fft_defaults'].update(
    data_root=str(TFDS_ROOT), prepared_data_root=str(PREPARED), image_aug=False,
    batch_size=1, grad_accumulation_steps=1, max_steps=1, save_steps=1,
    save_optimizer_state=True, eval_steps=0, debug_records=0)
config['fft_runs']['ava_vla'].update(
    checkpoint=str(ROOT / 'checkpoints/teacher/avavla-libero-4in1'),
    output_dir=str(OUTPUT))
if CONFIG.exists() and json.loads(CONFIG.read_text()) != config:
    raise ValueError('기존 1-step 설정이 다릅니다. OUTPUT/CONFIG 이름을 바꾸십시오.')
CONFIG.write_text(json.dumps(config, indent=2))
run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla',
     '--config', CONFIG, '--preflight'])

In [ ]:
CHECKPOINT = OUTPUT / 'step-0000001'
if not (CHECKPOINT / 'COMPLETE').is_file():
    run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', CONFIG])
metrics = sorted(OUTPUT.glob('metrics-*.jsonl'))
runs = sorted(OUTPUT.glob('run-*.json'))
if len(metrics) != 1 or len(runs) != 1:
    raise ValueError('격리된 새 OUTPUT에는 metrics/run manifest가 각각 하나여야 합니다.')
run([AVA_PY, 'scripts/train/check_fft_checkpoint.py',
     '--checkpoint', CHECKPOINT, '--data', PREPARED, '--config', CONFIG,
     '--metrics', metrics[0], '--run-manifest', runs[0],
     '--report', OUTPUT / 'reload_report.json'])

In [ ]:
report = json.loads((OUTPUT / 'reload_report.json').read_text())
summary = {
    'status': report['status'], 'gpu': report['gpu'],
    'step_time_seconds': report['training_metrics']['step_time_seconds'],
    'training_peak_allocated_gib': report['training_metrics']['peak_allocated_bytes'] / 2**30,
    'training_peak_reserved_gib': report['training_metrics']['peak_reserved_bytes'] / 2**30,
    'training_peak_process_ram_gib': report['training_metrics']['peak_process_rss_bytes'] / 2**30,
    'reload_peak_allocated_gib': report['reload_peak_allocated_bytes'] / 2**30,
    'reload_peak_process_ram_gib': report['reload_peak_process_rss_bytes'] / 2**30,
    'checkpoint_gib': report['checkpoint_bytes'] / 2**30,
    'loss': report['training_metrics']['loss'],
    'grad_norm': report['training_metrics']['grad_norm'],
    'gradient_diagnostics': report['gradient_diagnostics'],
    'action_finite': report['action_finite'],
    'training_finite_checks': report['training_finite_checks'],
    'scope': report['scope'],
    'ava_temporal_training': 'zero_context',
}
print(json.dumps(summary, indent=2))
print('원본 보고서:', OUTPUT / 'reload_report.json')